# Experiment analysis — constrained tagging vs. context-based span labeling

Everything here is computed from the per-generation prediction files (`Experiment_results_publication/*/*/Predictions/*.jsonl`), not from the result CSVs: an arm that was split into shards has one CSV per shard, and an F1 cannot be averaged over shards — the counts have to be pooled first.

1. Load and validate the predictions
2. F1 report tables (UNER and ESA-MT macro-averaged over their subsets)
3. Main comparison: constrained tagging vs. context-based vs. unconstrained tagging
4. Reference numbers from the literature and the Semin et al. comparison
5. Error analysis
6. Reasoning and decode-time cost
7. LaTeX rows for `paper/tables/`
8. Full-dataset UNER runs for the Semin et al. comparison

## 1. Load and validate the predictions

In [1]:
import sys, json, re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Repo root = the first parent that holds src/utils, so the notebook runs from anywhere in the repo.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

RES = ROOT / "Experiment_results_publication"            # the 112 arms, capped CoNLL/UNER, 3 seeds
RES_FULL = ROOT / "Experiment_results_publication_full"  # full UNER for the Semin et al. comparison
CACHE = ROOT / "Notebooks" / "CG_publication" / "cache"  # parsed predictions, so the JSONL is read only once
CACHE.mkdir(exist_ok=True)

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)

# Row order used in every table.
MODELS = ["gemma-4-E2B-it", "gemma-4-31B-it", "Qwen3-8B", "Qwen3.8-27B", "gpt-oss-20b", "gpt-oss-120b"]
ARMS = ["OFF", "ON", "low", "medium"]                    # reasoning arm; ON = model default, low/medium = effort
DATASETS = ["CoNLL", "ToxicSpans", "LegalQAEval", "UNER", "ESA-MT", "GEC"]   # column order of tab_main_results
APPROACHES = {"Constrained-Gen": "CG", "Context-Based": "CTX"}

# Datasets that are evaluated per subset and reported as the MACRO average over the subsets (Semin et al. protocol):
# UNER = 18 treebanks, ESA-MT = 24 language-domain files. Everything else is one pool.
MACRO_DATASETS = {"UNER": 18, "ESA-MT": 24}

# Examples per seed each arm must have. CoNLL and UNER are capped (--max-examples 1152 / 250 per treebank).
EXPECTED_N = {"CoNLL": 1152, "ToxicSpans": 1000, "LegalQAEval": 1206, "UNER": 3702, "ESA-MT": 867, "GEC": 504}

SEEDS = [42, 43, 44]
MAX_NEW_TOKENS = 18000   # every run used --max-new-tokens 18000; a generation of this length was cut off

# The three decoding methods that are compared. "mode" is the eval_mode of the JSONL.
METHODS = {"constrained": "Constrained tagging", "unconstrained": "Unconstrained tagging", "context": "Context-based"}

### Reading the prediction files

A prediction file is one arm (or one shard / one subset of an arm), with one JSON line per generation for all three seeds. `parse_file` reduces every line to counts, so the pooled metrics, the error decomposition and the failure modes can all be computed later without touching the JSONL again.

In [2]:
from seqeval.metrics.sequence_labeling import get_entities

_MODEL_RE = "|".join(re.escape(m) for m in sorted(MODELS, key=len, reverse=True))
_SHARD_RE = r"(?:_shard(?P<si>\d+)of(?P<sn>\d+))?"

# Prediction file: <prefix>[_shard<i>of<N>]_<model>_think_<bool>_sampling_<mode>_think<0|1>[_effort_<e>][_<processor>][_bs<B>].jsonl
PRED_RE = re.compile(
    rf"^(?P<prefix>.+?){_SHARD_RE}_(?P<model>{_MODEL_RE})_think_(?P<think>True|False)_sampling_"
    rf"(?P<mode>constrained|unconstrained|context)(?:_exact)?_think[01](?:_effort_(?P<effort>[a-z]+))?"
    rf"(?:_(?:token_aware|n\|a))?(?:_bs(?P<bs>\d+))?\.jsonl$")

# CSV file: <prefix>[_shard<i>of<N>]_<model>_<B>_BS_think<0|1>[_effort_<e>]_sampling.csv
CSV_RE = re.compile(
    rf"^(?P<prefix>.+?){_SHARD_RE}_(?P<model>{_MODEL_RE})_(?P<bs>\d+)_BS_think(?P<think>[01])"
    rf"(?:_effort_(?P<effort>[a-z]+))?_sampling\.csv$")


def arm_name(think, effort):
    """Reasoning arm: OFF, ON (model default effort), or the effort level."""
    if not think:
        return "OFF"
    return effort or "ON"


def list_prediction_files(root, datasets=DATASETS):
    """One row per prediction file that belongs to a FINISHED run.

    The Predictions folders also hold leftovers of differently-sharded runs that timed out. A shard set is only trusted
    when its CSVs exist (the CSV is written when a process finishes) and all N shards are there. UNER and ESA-MT are not
    index-sharded -- each subset has exactly one file per arm, overwritten by reruns -- so they need no such check.
    """
    rows = []
    for dataset in datasets:
        for approach_dir, approach in APPROACHES.items():
            base = root / dataset / approach_dir
            if not base.is_dir():
                continue

            # Which shard sets are complete, per arm: {(model, arm, bs): {N: {shard ids}}}
            shard_sets = defaultdict(lambda: defaultdict(set))
            written = defaultdict(float)   # newest CSV of each shard set
            for f in (base / "Csv").glob("*.csv"):
                m = CSV_RE.match(f.name)
                assert m, f"unparsed CSV name: {f.name}"
                key = (m["model"], arm_name(m["think"] == "1", m["effort"]), int(m["bs"]))
                shard_sets[key][m["sn"]].add(m["si"])
                written[key, m["sn"]] = max(written[key, m["sn"]], f.stat().st_mtime)
            complete = {}
            for key, sets in shard_sets.items():
                done = [sn for sn, ids in sets.items() if sn is None or len(ids) == int(sn)]
                if len(done) > 1:   # an arm that was re-run with another shard count: the newest run counts
                    print(f"[note] {dataset} {approach} {key}: {len(done)} complete shard sets, using the newest")
                    done = [max(done, key=lambda sn: written[key, sn])]
                complete[key] = set(done)

            for f in sorted((base / "Predictions").glob("*.jsonl")):
                m = PRED_RE.match(f.name)
                assert m, f"unparsed prediction name: {f.name}"
                key = (m["model"], arm_name(m["think"] == "True", m["effort"]), int(m["bs"] or 1))
                if dataset not in MACRO_DATASETS and m["sn"] not in complete.get(key, set()):
                    continue   # leftover of an unfinished / superseded shard set
                rows.append({"dataset": dataset, "approach": approach, "mode": m["mode"], "model": key[0],
                             "arm": key[1], "bs": key[2], "shard": m["si"], "n_shards": m["sn"], "path": str(f)})
    return pd.DataFrame(rows)


def _overlaps(a, b):
    """Do two spans share a character? A zero-length span (MultiGEC's M) counts when it touches the other one."""
    if a["start"] == a["end"] or b["start"] == b["end"]:
        return a["start"] <= b["end"] and b["start"] <= a["end"]
    return max(a["start"], b["start"]) < min(a["end"], b["end"])


def span_errors(gold, pred):
    """Split the predicted and gold spans of one example into the error classes of the error analysis.

    Every predicted span lands in exactly one of:
      exact       -- same boundaries and label as a gold span
      label       -- same boundaries as a gold span, different label
      boundary    -- overlaps a gold span with the same label, boundaries differ
      bound_label -- overlaps gold spans, none of them with the same label
      spurious    -- overlaps no gold span
    and every gold span that no predicted span overlaps is `missed`.
    Also returns the (gold label, predicted label) pairs of the boundary-exact spans, for the confusion matrix.
    """
    out = Counter()
    pairs = []
    gold_by_bounds = {(g["start"], g["end"]): g["label"] for g in gold}
    gold_keys = {(g["start"], g["end"], g["label"]) for g in gold}
    for p in pred:
        bounds = (p["start"], p["end"])
        if (*bounds, p["label"]) in gold_keys:
            out["exact"] += 1
            pairs.append((p["label"], p["label"]))
        elif bounds in gold_by_bounds:
            out["label"] += 1
            pairs.append((gold_by_bounds[bounds], p["label"]))
        else:
            hit = [g for g in gold if _overlaps(g, p)]
            if not hit:
                out["spurious"] += 1
            elif any(g["label"] == p["label"] for g in hit):
                out["boundary"] += 1
            else:
                out["bound_label"] += 1
    out["missed"] = sum(1 for g in gold if not any(_overlaps(g, p) for p in pred))
    return out, pairs


def outcome(rec, mode):
    """One outcome label per generation.

    Context-based runs already carry Semin et al.'s 11-way `error_type`. Tagging runs get the matching classification
    here, most severe first: the text-copy failures take the place of their parsing / span-matching errors.
    """
    if mode == "context":
        return rec["error_type"]
    if not (rec.get("raw_output") or "").strip():
        return "empty_response"
    if rec["reasoning_enabled"] and not rec["reasoning_skipped"] and not rec["found_reasoning_end"]:
        return "reasoning_unterminated"      # the whole budget went into reasoning, no answer at all
    if rec["wrong_text"]:
        # The answer started but does not reproduce the input. At the token cap it is a truncated copy; when the model
        # skipped its reasoning block the answer began before the constraint was switched on.
        if rec["num_output_tokens"] >= MAX_NEW_TOKENS:
            return "max_tokens_exceeded"
        return "reasoning_skipped" if rec["reasoning_skipped"] else "wrong_text"
    if not rec["pred_spans"]:
        return "empty_prediction"
    return "success"


def parse_file(meta):
    """Read one prediction JSONL into compact per-generation rows (+ label-confusion counts). None if it is corrupt.

    One JSONL line is one generation: one example, or one batch of `bs` sentences for the NER runs. Everything the
    analysis needs is reduced to counts here, so the raw outputs and token ids never have to be kept in memory.
    """
    rows, confusion = [], Counter()
    with open(meta["path"], encoding="utf-8") as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                return None, None   # several processes wrote into this file at once, nothing in it can be trusted
            gold, pred = rec["gold_spans"], rec["pred_spans"]
            errs, pairs = span_errors(gold, pred)
            confusion.update(pairs)

            # Subset for the macro average: treebank for UNER, language-domain file for ESA-MT.
            subset = rec["dataset"].split("_", 1)[1] if meta["dataset"] in MACRO_DATASETS else "all"
            ids = rec.get("example_ids") or [rec["example_id"]]
            stats = rec.get("match_stats") or {}

            row = {
                "subset": subset, "seed": rec["seed"], "key": rec["key"], "n_examples": len(ids),
                "example_id": "|".join(map(str, ids)),
                # Semin et al. character-overlap counts, micro-pooled later
                "hard_overlap": rec["semin_hard_overlap"], "hard_pred": rec["semin_hard_predicted"],
                "hard_gold": rec["semin_hard_gold"], "soft_overlap": rec["semin_soft_overlap"],
                "soft_pred": rec["semin_soft_predicted"], "soft_gold": rec["semin_soft_gold"],
                # per-example char F1 (label-agnostic) and per-example hard F1, where the script logged them
                "char_f1": rec.get("char_f1", np.nan), "macro_hard_f1": rec.get("macro_hard_f1", np.nan),
                # span counts, lengths and the error decomposition
                "n_gold": len(gold), "n_pred": len(pred),
                "gold_chars": sum(g["end"] - g["start"] for g in gold),
                "pred_chars": sum(p["end"] - p["start"] for p in pred),
                "gold_empty": sum(g["start"] == g["end"] for g in gold),
                "pred_empty": sum(p["start"] == p["end"] for p in pred),
                **{f"e_{k}": errs.get(k, 0) for k in ("exact", "label", "boundary", "bound_label", "spurious", "missed")},
                # failure modes
                "outcome": outcome(rec, meta["mode"]),
                "wrong_text": rec.get("wrong_text", 0),
                "hit_cap": int(rec["num_output_tokens"] >= MAX_NEW_TOKENS),
                "unterminated": int(rec["reasoning_enabled"] and not rec["reasoning_skipped"]
                                    and not rec["found_reasoning_end"]),
                "skipped": int(rec["reasoning_skipped"]),   # reasoning arm, but the model went straight to the answer
                # context-based matching failures, per proposed item
                "ctx_items": stats.get("processed_entities", 0) + stats.get("invalid_entity_format", 0),
                "ctx_not_found": stats.get("unlocated_entities", 0),
                "ctx_invalid_label": stats.get("invalid_label_count", 0),
                # cost
                "out_tokens": rec["num_output_tokens"], "reasoning_tokens": rec["num_reasoning_tokens"],
                "answer_tokens": rec["num_answer_tokens"], "gen_seconds": rec["generation_seconds"],
                "input_chars": len(rec["input_text"]),
            }
            # Entity-level (seqeval) counts for the NER datasets, from the BIO tags the eval script scored.
            if "gold_tags" in rec:
                g_ent, p_ent = set(get_entities(rec["gold_tags"])), set(get_entities(rec["pred_tags"]))
                row.update(ent_tp=len(g_ent & p_ent), ent_pred=len(p_ent), ent_gold=len(g_ent))
            rows.append(row)

    df = pd.DataFrame(rows)
    for col in ("dataset", "approach", "mode", "model", "arm", "bs"):
        df[col] = meta[col]
    conf = pd.DataFrame([{**{c: meta[c] for c in ("dataset", "approach", "mode", "model", "arm", "bs")},
                          "gold_label": g, "pred_label": p, "n": n} for (g, p), n in confusion.items()])
    return df, conf


def load_predictions(root, cache_name, datasets=DATASETS, refresh=False):
    """All finished runs under `root` as one per-generation DataFrame. Cached as parquet; refresh=True re-reads."""
    pred_cache, conf_cache = CACHE / f"{cache_name}_predictions.parquet", CACHE / f"{cache_name}_confusion.parquet"
    if pred_cache.exists() and not refresh:
        return pd.read_parquet(pred_cache), pd.read_parquet(conf_cache)

    files = list_prediction_files(root, datasets)
    frames, confs, corrupt = [], [], []
    for i, meta in enumerate(files.to_dict("records")):
        df, conf = parse_file(meta)
        if df is None:
            corrupt.append(meta)
            continue
        frames.append(df)
        confs.append(conf)
        if (i + 1) % 200 == 0:
            print(f"  parsed {i + 1}/{len(files)} files")
    preds, confusion = pd.concat(frames, ignore_index=True), pd.concat(confs, ignore_index=True)

    # An arm with a corrupt shard is dropped as a whole -- a partial arm would silently give a wrong pooled score.
    arm_cols = ["dataset", "approach", "mode", "model", "arm", "bs"]
    for arm in pd.DataFrame(corrupt, columns=files.columns)[arm_cols].drop_duplicates().to_dict("records"):
        print(f"[corrupt, arm dropped] {arm}")
        preds = preds[~(preds[arm_cols] == pd.Series(arm)).all(axis=1)]
        confusion = confusion[~(confusion[arm_cols] == pd.Series(arm)).all(axis=1)]
    confusion = confusion.groupby([c for c in confusion.columns if c != "n"], as_index=False)["n"].sum()
    preds.to_parquet(pred_cache)
    confusion.to_parquet(conf_cache)
    return preds, confusion

In [3]:
# First run parses ~5 GB of JSONL (a few minutes); afterwards the parquet cache is read. Use refresh=True after new runs.
preds, confusion = load_predictions(RES, "main")

ARM = ["dataset", "approach", "mode", "model", "arm", "bs"]   # one row of any result table

# Sanity check of what was loaded: every arm must have 3 seeds, the expected number of examples per seed, all subsets,
# and no example twice (a duplicate would mean two shard sets of the same arm were mixed together).
check = (preds.groupby(ARM + ["seed"])
         .agg(n=("n_examples", "sum"), subsets=("subset", "nunique"),
              duplicates=("example_id", lambda s: s.duplicated().sum()))
         .reset_index())
check["expected_n"] = check["dataset"].map(EXPECTED_N)
check["expected_subsets"] = check["dataset"].map(MACRO_DATASETS).fillna(1).astype(int)
bad = check[(check.n != check.expected_n) | (check.subsets != check.expected_subsets) | (check.duplicates > 0)]

arms = preds[["dataset", "approach", "model", "arm", "bs"]].drop_duplicates()
seeds_per_arm = preds.groupby(ARM)["seed"].nunique()
print(f"{len(preds):,} generations, {len(arms)} arms "
      f"({(arms.approach == 'CG').sum()} CG + {(arms.approach == 'CTX').sum()} context-based)")
print(f"arms without all 3 seeds: {(seeds_per_arm != 3).sum()}")
print(f"arm-seeds with a wrong example count, missing subset or duplicate: {len(bad)}")
bad

607,614 generations, 110 arms (80 CG + 30 context-based)
arms without all 3 seeds: 0
arm-seeds with a wrong example count, missing subset or duplicate: 0


,dataset,approach,mode,model,arm,bs,seed,n,subsets,duplicates,expected_n,expected_subsets


### Metrics

| metric | what it is | pooled how |
|---|---|---|
| `hard_*` | Semin et al. character overlap, label must match | micro: counts summed over the pool |
| `soft_*` | same, label ignored | micro |
| `ent_*` | entity-level F1 (seqeval), CoNLL and UNER only | micro |
| `span_*` | exact span match (boundaries and label), all datasets | micro |
| `char_f1` | our per-example char-F1, label-agnostic | mean over examples |

A *pool* is one seed of one dataset — or of one subset for UNER and ESA-MT, whose score is then the macro average over the subsets. The reported number is the mean over the three seeds, ± the standard deviation over seeds.

In [4]:
from utils.semin_metrics import f1_from_counts   # Semin et al.'s own function, imported from their repo

# Counters that are summed inside a pool before any F1 is computed.
COUNT_COLS = ["hard_overlap", "hard_pred", "hard_gold", "soft_overlap", "soft_pred", "soft_gold",
              "ent_tp", "ent_pred", "ent_gold", "e_exact", "n_pred", "n_gold", "n_examples"]

# (metric prefix, overlap/TP column, predicted column, gold column)
F1_KINDS = [("hard", "hard_overlap", "hard_pred", "hard_gold"),    # Semin char overlap, label must match
            ("soft", "soft_overlap", "soft_pred", "soft_gold"),    # Semin char overlap, label ignored
            ("ent", "ent_tp", "ent_pred", "ent_gold"),             # seqeval entity-level F1 (NER datasets only)
            ("span", "e_exact", "n_pred", "n_gold")]               # exact span match: boundaries AND label
METRICS = [f"{kind}_{m}" for kind, *_ in F1_KINDS for m in ("p", "r", "f1")] + ["char_f1"]


def score_pools(df, by):
    """Micro scores of every pool defined by `by`: counts are summed over the pool, then turned into P/R/F1.

    This is what combines shards correctly -- a per-shard F1 cannot be averaged into the F1 of the whole dataset, the
    counts have to be pooled first. `char_f1` is the exception: it is our per-example macro char-F1, so it is a mean.
    """
    pools = df.groupby(by, observed=True)[COUNT_COLS].sum(min_count=1)
    pools["char_f1"] = df.groupby(by, observed=True)["char_f1"].mean()
    for kind, tp, n_pred, n_gold in F1_KINDS:
        scores = [f1_from_counts(*row) if not np.isnan(row[0]) else {"precision": np.nan, "recall": np.nan, "f1": np.nan}
                  for row in pools[[tp, n_pred, n_gold]].itertuples(index=False)]
        pools[f"{kind}_p"] = [s["precision"] for s in scores]
        pools[f"{kind}_r"] = [s["recall"] for s in scores]
        pools[f"{kind}_f1"] = [s["f1"] for s in scores]
    return pools.reset_index()


def aggregate(df):
    """Per-subset, per-seed and final (mean and std over the 3 seeds) scores for every arm.

    CoNLL / ToxicSpans / LegalQAEval / GEC: one pool per seed (subset == "all"), all shards combined.
    UNER / ESA-MT: one pool per subset per seed, then the MACRO average over the 18 treebanks / 24 files -- this is the
    number Semin et al. report, and it differs from pooling all subsets into one.
    """
    per_subset = score_pools(df, ARM + ["subset", "seed"])
    per_seed = per_subset.groupby(ARM + ["seed"], observed=True)[METRICS].mean().reset_index()
    final = per_seed.groupby(ARM, observed=True)[METRICS].agg(["mean", "std"])
    final.columns = [f"{m}_{stat}" for m, stat in final.columns]
    return per_subset, per_seed, final.reset_index()


def pm(mean, std, digits=2):
    """'mean ± std' in percent, the format of the result CSVs."""
    if pd.isna(mean):
        return "--"
    return f"{100 * mean:.{digits}f} ± {100 * std:.{digits}f}"


def order(df):
    """Sort rows in the fixed dataset / model / arm order instead of alphabetically."""
    df = df.copy()
    for col, cats in (("dataset", DATASETS), ("model", MODELS), ("arm", ARMS), ("mode", list(METHODS))):
        if col in df.columns:
            df[col] = pd.Categorical(df[col], cats, ordered=True)
    return df.sort_values([c for c in ("dataset", "mode", "model", "arm", "bs") if c in df.columns])


per_subset, per_seed, final = aggregate(preds)
print(f"{len(final)} result rows: " + ", ".join(f"{n} {mode}" for mode, n in final["mode"].value_counts().items()))

190 result rows: 80 constrained, 80 unconstrained, 30 context


In [5]:
# Cross-check against the numbers the eval scripts wrote themselves. Only arms that ran as ONE process can be compared
# (single CSV, no shards) -- for those the recomputed value must equal the CSV, which validates the loader and the
# metric code before they are trusted on the sharded arms where the CSVs cannot be combined.
rows = []
for f in sorted(RES.glob("*/*/Csv/*.csv")):
    m = CSV_RE.match(f.name)
    dataset, approach = f.parts[-4], APPROACHES.get(f.parts[-3])
    if approach is None or m["sn"] is not None or dataset in MACRO_DATASETS:
        continue
    for r in pd.read_csv(f).to_dict("records"):
        mine = final[(final.dataset == dataset) & (final.approach == approach) & (final["mode"] == r["eval_mode"])
                     & (final.model == m["model"]) & (final.bs == int(m["bs"]))
                     & (final.arm == arm_name(m["think"] == "1", m["effort"]))].iloc[0]
        rows.append({"dataset": dataset, "mode": r["eval_mode"], "model": m["model"], "bs": int(m["bs"]),
                     "csv_hard_f1": r["semin_hard_f1_report"], "recomputed_hard_f1": pm(mine.hard_f1_mean, mine.hard_f1_std),
                     "csv_f1": r.get("f1_report", r.get("char_f1_report")),
                     "recomputed_f1": pm(*(mine[["ent_f1_mean", "ent_f1_std"]] if dataset == "CoNLL"
                                           else mine[["char_f1_mean", "char_f1_std"]]))})
csv_check = pd.DataFrame(rows)

# Compare the means numerically: the CSV rounds each seed before averaging, so the last digit may differ by 0.01.
mean_of = lambda col: csv_check[col].str.split(" ± ").str[0].astype(float)
csv_check["hard_diff"] = (mean_of("csv_hard_f1") - mean_of("recomputed_hard_f1")).abs()
csv_check["f1_diff"] = (mean_of("csv_f1") - mean_of("recomputed_f1")).abs()
print(f"{len(csv_check)} unsharded rows compared, largest difference: "
      f"hard F1 {csv_check.hard_diff.max():.2f}, entity / char F1 {csv_check.f1_diff.max():.2f} points")
csv_check[(csv_check.hard_diff > 0.02) | (csv_check.f1_diff > 0.02)]

27 unsharded rows compared, largest difference: hard F1 0.00, entity / char F1 0.01 points


,dataset,mode,model,bs,csv_hard_f1,recomputed_hard_f1,csv_f1,recomputed_f1,hard_diff,f1_diff


## 2. F1 report tables

In [6]:
def report(metric="hard_f1", mode="constrained", bs=1, table=None):
    """Arms x datasets table of one metric for one decoding mode: mean ± std over the 3 seeds, in %."""
    d = order((final if table is None else table).query("mode == @mode and bs == @bs"))
    d["cell"] = [pm(m, s) for m, s in zip(d[f"{metric}_mean"], d[f"{metric}_std"])]
    wide = d.pivot_table(index=["model", "arm"], columns="dataset", values="cell", aggfunc="first", observed=True)
    return wide.reindex(columns=[c for c in DATASETS if c in wide.columns]).fillna("")


# Headline metric: Semin et al. hard F1 (character overlap, label must match). UNER and ESA-MT are macro averages.
for mode, name in METHODS.items():
    print(f"\n=== {name} -- hard F1, batch size 1 ===")
    print(report("hard_f1", mode).to_string())


=== Constrained tagging -- hard F1, batch size 1 ===
dataset                       CoNLL    ToxicSpans   LegalQAEval          UNER        ESA-MT           GEC
model          arm                                                                                       
gemma-4-E2B-it OFF     56.44 ± 0.58  13.24 ± 0.57  11.25 ± 0.35  58.58 ± 0.59   4.60 ± 0.95   8.25 ± 0.73
               ON      68.00 ± 0.79  13.39 ± 1.03  10.91 ± 0.42  64.29 ± 0.52   7.80 ± 1.41   8.97 ± 0.46
gemma-4-31B-it OFF     84.69 ± 0.13  25.30 ± 0.19  55.55 ± 0.34  88.68 ± 0.14  13.96 ± 0.16  50.65 ± 1.41
Qwen3-8B       OFF     63.29 ± 0.50  28.74 ± 0.33  19.64 ± 0.74  70.30 ± 0.35  13.62 ± 0.07  15.71 ± 0.35
Qwen3.8-27B    OFF     72.76 ± 0.09  17.41 ± 0.68  41.81 ± 2.19  59.78 ± 0.89   9.19 ± 1.04  24.54 ± 0.89
               low     80.39 ± 0.57  20.02 ± 0.51  51.75 ± 0.88  54.36 ± 0.62   9.46 ± 0.36  49.92 ± 0.18
gpt-oss-20b    low     69.90 ± 1.04  20.51 ± 1.01  42.94 ± 1.29  75.72 ± 1.36  12.87 ± 0.12  24.54

In [7]:
# Same table for soft F1 (label ignored) -- the gap to hard F1 is the share of the error that is a wrong LABEL on a
# correctly located span. It is zero by definition on the single-label tasks (ToxicSpans, LegalQAEval).
print("=== Constrained tagging -- soft F1, batch size 1 ===")
print(report("soft_f1", "constrained").to_string())

# Task-native metrics, the ones the original benchmarks use: entity-level F1 (seqeval) for the NER datasets and the
# per-example macro char-F1 for the rest. These are the numbers to put next to fine-tuned baselines from the literature.
native = final.copy()
is_ner = native.dataset.isin(["CoNLL", "UNER"])
native["native_f1_mean"] = np.where(is_ner, native.ent_f1_mean, native.char_f1_mean)
native["native_f1_std"] = np.where(is_ner, native.ent_f1_std, native.char_f1_std)
for mode, name in METHODS.items():
    print(f"\n=== {name} -- task-native F1 (entity F1 for CoNLL/UNER, char-F1 otherwise), batch size 1 ===")
    print(report("native_f1", mode, table=native).to_string())

# Batch size 64: only the two NER datasets were run with it (64 sentences tagged in one generation).
print("\n=== Constrained tagging -- hard F1, batch size 64 ===")
print(report("hard_f1", "constrained", bs=64).to_string())
print("\n=== Context-based -- hard F1, batch size 64 ===")
print(report("hard_f1", "context", bs=64).to_string())

=== Constrained tagging -- soft F1, batch size 1 ===
dataset                       CoNLL    ToxicSpans   LegalQAEval          UNER        ESA-MT           GEC
model          arm                                                                                       
gemma-4-E2B-it OFF     76.53 ± 0.99  13.24 ± 0.57  11.25 ± 0.35  66.26 ± 0.37   9.83 ± 1.01  12.85 ± 0.35
               ON      81.88 ± 0.64  13.39 ± 1.03  10.91 ± 0.42  69.10 ± 0.45  17.28 ± 1.03  16.14 ± 0.32
gemma-4-31B-it OFF     94.57 ± 0.31  25.30 ± 0.19  55.55 ± 0.34  92.22 ± 0.26  34.04 ± 0.85  53.99 ± 1.37
Qwen3-8B       OFF     78.29 ± 0.53  28.74 ± 0.33  19.64 ± 0.74  76.29 ± 0.16  20.49 ± 0.29  20.23 ± 0.19
Qwen3.8-27B    OFF     81.49 ± 0.54  17.41 ± 0.68  41.81 ± 2.19  63.24 ± 1.05  22.76 ± 0.39  31.14 ± 0.73
               low     88.48 ± 0.51  20.02 ± 0.51  51.75 ± 0.88  56.72 ± 0.64  25.87 ± 0.50  52.66 ± 0.29
gpt-oss-20b    low     81.95 ± 0.70  20.51 ± 1.01  42.94 ± 1.29  80.38 ± 1.11  25.67 ± 0.47  28.08 

### Macro average over the UNER treebanks and the ESA-MT files

In [8]:
# Macro (average of the per-subset F1s, what is reported) vs micro (all subsets pooled into one) on UNER and ESA-MT.
# `final` already holds the macro; the micro is recomputed here by pooling without the subset in the group key.
multi = preds[preds.dataset.isin(MACRO_DATASETS)]
micro = (score_pools(multi, ARM + ["seed"]).groupby(ARM, observed=True)["hard_f1"].mean().rename("micro_hard_f1"))
macro = final.set_index(ARM)["hard_f1_mean"].rename("macro_hard_f1")
macro_vs_micro = pd.concat([macro, micro], axis=1, join="inner").reset_index()
macro_vs_micro["macro - micro"] = macro_vs_micro.macro_hard_f1 - macro_vs_micro.micro_hard_f1

show = order(macro_vs_micro.query("mode == 'constrained' and bs == 1")).drop(columns=["approach", "mode", "bs"])
show[["macro_hard_f1", "micro_hard_f1", "macro - micro"]] *= 100
print(show.round(2).to_string(index=False))

dataset          model    arm  macro_hard_f1  micro_hard_f1  macro - micro
   UNER gemma-4-E2B-it    OFF          58.58          59.35          -0.77
   UNER gemma-4-E2B-it     ON          64.29          64.43          -0.14
   UNER gemma-4-31B-it    OFF          88.68          88.28           0.39
   UNER       Qwen3-8B    OFF          70.30          68.00           2.31
   UNER    Qwen3.8-27B    OFF          59.78          61.14          -1.36
   UNER    Qwen3.8-27B    low          54.36          55.62          -1.26
   UNER    gpt-oss-20b    low          75.72          75.39           0.33
   UNER    gpt-oss-20b medium          86.20          85.36           0.84
   UNER   gpt-oss-120b    low          79.52          79.36           0.16
   UNER   gpt-oss-120b medium          89.27          88.44           0.83
 ESA-MT gemma-4-E2B-it    OFF           4.60           4.04           0.55
 ESA-MT gemma-4-E2B-it     ON           7.80           8.55          -0.75
 ESA-MT gemma-4-31B-it   

In [9]:
def subset_table(dataset, metric="hard_f1", mode="constrained", bs=1):
    """Per-subset scores (mean over seeds, %) with the macro average as the last row -- one column per arm."""
    d = order(per_subset.query("dataset == @dataset and mode == @mode and bs == @bs"))
    wide = d.pivot_table(index="subset", columns=["model", "arm"], values=metric, aggfunc="mean", observed=True) * 100
    wide.loc["MACRO AVG"] = wide.mean()
    return wide.round(1)


print("UNER, constrained tagging, hard F1 per treebank (250 examples max per treebank):")
print(subset_table("UNER").to_string())
print("\nESA-MT, constrained tagging, hard F1 per language pair and domain:")
print(subset_table("ESA-MT").to_string())

UNER, constrained tagging, hard F1 per treebank (250 examples max per treebank):
model        gemma-4-E2B-it       gemma-4-31B-it Qwen3-8B Qwen3.8-27B       gpt-oss-20b        gpt-oss-120b       
arm                     OFF    ON            OFF      OFF         OFF   low         low medium          low medium
subset                                                                                                            
ceb_gja                48.3  57.3           93.5     77.7        63.0  55.0        68.8   85.3         77.1   93.3
da_ddt                 55.7  61.0           90.8     65.0        63.3  56.1        73.1   86.7         78.6   89.9
de_pud                 63.8  68.7           84.8     64.8        63.8  52.7        71.4   82.4         74.6   85.5
en_ewt                 75.7  75.4           89.9     72.9        66.1  68.9        78.1   87.1         77.0   88.1
en_pud                 72.4  78.1           86.3     69.9        56.3  57.0        72.1   84.4         80.0   87.0

## 3. Main comparison: the two methods on the six tasks

In [10]:
# The two methods side by side, on the arms that were run for BOTH (the context-based baseline has 4 arms per dataset).
# Unconstrained tagging is the third column: same prompt and markup as ours, only the constraint switched off.
def method_comparison(metric="hard_f1", bs=1):
    d = final.query("bs == @bs").pivot_table(index=["dataset", "model", "arm"], columns="mode",
                                              values=f"{metric}_mean", observed=True) * 100
    d = d.dropna(subset=["context"])                      # keep the shared arms only
    d["CG - context"] = d["constrained"] - d["context"]
    d["CG - unconstrained"] = d["constrained"] - d["unconstrained"]
    return order(d.reset_index())


comparison = method_comparison("hard_f1")
wide = comparison.pivot_table(index=["model", "arm"], columns="dataset", observed=True,
                              values=["constrained", "context", "unconstrained", "CG - context"])
wide = wide.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product(
    [DATASETS, ["constrained", "context", "CG - context", "unconstrained"]]))
print("Hard F1 (%), batch size 1 -- constrained tagging vs the context-based baseline vs unconstrained tagging")
for dataset in DATASETS:
    print(f"\n--- {dataset} ---")
    print(wide[dataset].round(1).to_string())

# How often does each method win, and by how much on average per dataset?
summary = comparison.groupby("dataset", observed=True).agg(
    arms=("constrained", "size"),
    cg_wins=("CG - context", lambda s: (s > 0).sum()),
    mean_cg=("constrained", "mean"), mean_context=("context", "mean"), mean_unconstrained=("unconstrained", "mean"),
    mean_delta_vs_context=("CG - context", "mean"), mean_delta_vs_unconstrained=("CG - unconstrained", "mean"))
print("\nPer dataset, averaged over the shared arms:")
print(summary.round(2).to_string())

Hard F1 (%), batch size 1 -- constrained tagging vs the context-based baseline vs unconstrained tagging

--- CoNLL ---
                    constrained  context  CG - context  unconstrained
model          arm                                                   
gemma-4-E2B-it OFF         56.4     71.3         -14.9            8.7
               ON          68.0     72.5          -4.5           47.6
Qwen3.8-27B    OFF         72.8     84.0         -11.2           69.9
gpt-oss-20b    low         69.9     73.3          -3.4           59.8

--- ToxicSpans ---
                    constrained  context  CG - context  unconstrained
model          arm                                                   
gemma-4-E2B-it OFF         13.2     32.5         -19.2           28.7
               ON          13.4     29.4         -16.0           27.6
Qwen3.8-27B    OFF         17.4     21.7          -4.3           23.3
gpt-oss-20b    low         20.5     30.2          -9.6           23.7

--- LegalQAEval ---


## 4. Reference numbers

In [11]:
# Published reference numbers. None of them is a like-for-like comparison: they are fine-tuned on the training set, and
# CoNLL / UNER are evaluated here on a capped sample. They only anchor the scale of our zero-shot numbers.
#
# UNER: XLM-R large baselines of the UNER paper (Table 6, micro entity F1). 9 treebanks have a training set and are
# fine-tuned in-language; the PUD and the low-resource test sets are zero-shot transfer from en_ewt.
UNER_XLMR = {"da_ddt": 87.8, "en_ewt": 85.8, "hr_set": 95.0, "sk_snk": 85.5, "pt_bosque": 90.4, "sr_set": 96.2,
             "sv_talbanken": 88.3, "zh_gsd": 89.6, "zh_gsdsimp": 89.4,                       # in-language
             "de_pud": 81.4, "en_pud": 80.5, "pt_pud": 84.8, "ru_pud": 71.5, "sv_pud": 87.2, "zh_pud": 41.0,
             "ceb_gja": 78.9, "tl_trg": 93.6, "tl_ugnayan": 78.3}                            # transfer from en_ewt
assert set(UNER_XLMR) == set(preds.query("dataset == 'UNER'").subset.unique())

LITERATURE = pd.DataFrame([
    {"dataset": "CoNLL", "system": "ACE + document context (Wang et al., 2021)", "metric": "entity F1", "score": 94.6,
     "note": "fine-tuned, full test set (3,453 sentences)"},
    {"dataset": "CoNLL", "system": "LUKE (Yamada et al., 2020)", "metric": "entity F1", "score": 94.3,
     "note": "fine-tuned, full test set"},
    {"dataset": "ToxicSpans", "system": "HITSZ-HLT (Zhu et al., 2021), SemEval-2021 Task 5 winner",
     "metric": "char F1 (per-example macro)", "score": 70.83,
     "note": "fine-tuned ensemble, official 2,000-post test set; ours is the 1,000-post heegyu/toxic-spans test split"},
    {"dataset": "UNER", "system": "XLM-R large (Mayhew et al., 2024)", "metric": "entity F1, macro over 18 treebanks",
     "score": round(np.mean(list(UNER_XLMR.values())), 1),
     "note": "9 treebanks fine-tuned in-language, 9 zero-shot from en_ewt; full test sets"},
    {"dataset": "LegalQAEval", "system": "--", "metric": "--", "score": np.nan,
     "note": "Isaacus report Matthews correlation of answerable/unanswerable, no span-overlap F1 is published"},
    {"dataset": "ESA-MT", "system": "Semin et al. (2026)", "metric": "hard F1", "score": np.nan, "note": "see next section"},
    {"dataset": "GEC", "system": "Semin et al. (2026)", "metric": "hard F1", "score": np.nan, "note": "see next section"},
])

# Our best constrained arm per dataset on the same metric (batch size 1), to set next to the published number.
best = (order(native.query("mode == 'constrained' and bs == 1"))
        .sort_values("native_f1_mean", ascending=False).groupby("dataset", observed=True).head(1))
best["ours_best_CG"] = [f"{pm(m, s)}  ({model}, {arm})"
                        for m, s, model, arm in zip(best.native_f1_mean, best.native_f1_std, best.model, best.arm)]
print(LITERATURE.merge(best[["dataset", "ours_best_CG"]], on="dataset", how="left").to_string(index=False))

    dataset                                                   system                             metric  score                                                                                                    note                         ours_best_CG
      CoNLL               ACE + document context (Wang et al., 2021)                          entity F1  94.60                                                             fine-tuned, full test set (3,453 sentences)  82.40 ± 0.54  (gemma-4-31B-it, OFF)
      CoNLL                               LUKE (Yamada et al., 2020)                          entity F1  94.30                                                                               fine-tuned, full test set  82.40 ± 0.54  (gemma-4-31B-it, OFF)
 ToxicSpans HITSZ-HLT (Zhu et al., 2021), SemEval-2021 Task 5 winner        char F1 (per-example macro)  70.83 fine-tuned ensemble, official 2,000-post test set; ours is the 1,000-post heegyu/toxic-spans test split   38.79 ± 0.92  (gemma-4-E2B-

### Semin et al. (2026), same models

In [12]:
# Semin et al. ship every run of their paper in results.csv (one row per model x method x dataset file x seed), so the
# published numbers are recomputed here instead of being typed in. Their protocol: 5 seeds, UNER = macro over the 18
# treebanks, ESA-MT = macro over the 24 files. TAG in their paper is the `xml` method = unconstrained tagging.
semin = pd.read_csv(ROOT / "data/semin/raw/span_labeling/results/results.csv")
semin = semin[semin.model.isin(["google/gemma-4-31B-it", "Qwen/Qwen3-8B"]) & ~semin.thinking
              & semin.dataset_type.isin(["ner", "multigec", "wmt"])]
semin["model"] = semin.model.str.split("/").str[-1]
semin["dataset"] = semin.dataset_type.map({"ner": "UNER", "multigec": "GEC", "wmt": "ESA-MT"})

semin_f1 = (semin.groupby(["model", "method_name", "dataset", "seed"])[["hard_f1", "soft_f1"]].mean()   # macro over files
            .groupby(["model", "method_name", "dataset"]).agg(["mean", "std"]))                        # over 5 seeds
semin_f1.columns = [f"{m}_{s}" for m, s in semin_f1.columns]
semin_f1 = semin_f1.reset_index()

# Their tagging method, and their best method per model and dataset (whichever of the 11 it is).
semin_tag = semin_f1.query("method_name == 'xml'").assign(row="Semin et al. TAG (unconstrained)")
semin_best = (semin_f1.sort_values("hard_f1_mean", ascending=False).groupby(["model", "dataset"]).head(1)
              .assign(row=lambda d: "Semin et al. best method (" + d.method_name + ")"))

# Ours on the same two models, reasoning off, batch size 1. GEC and ESA-MT are the full datasets already; UNER is the
# 250-per-treebank cap here and is replaced by the full-dataset run in the last section once it has finished.
ours = final.query("model in ['gemma-4-31B-it', 'Qwen3-8B'] and arm == 'OFF' and bs == 1 "
                   "and dataset in ['UNER', 'GEC', 'ESA-MT'] and approach == 'CG'").copy()
ours["row"] = ours["mode"].map({"constrained": "Ours, constrained tagging", "unconstrained": "Ours, unconstrained tagging"})

published = pd.concat([semin_tag, semin_best, ours])[["model", "row", "dataset", "hard_f1_mean", "hard_f1_std",
                                                      "soft_f1_mean", "soft_f1_std"]]
published["hard F1"] = [pm(m, s, 1) for m, s in zip(published.hard_f1_mean, published.hard_f1_std)]
print("Hard F1 (%), reasoning off. UNER for ours: capped at 250 examples per treebank, 3 seeds (theirs: full, 5 seeds)")
print(published.pivot(index=["model", "row"], columns="dataset", values="hard F1")[["UNER", "GEC", "ESA-MT"]].to_string())

Hard F1 (%), reasoning off. UNER for ours: capped at 250 examples per treebank, 3 seeds (theirs: full, 5 seeds)
dataset                                                                      UNER         GEC      ESA-MT
model          row                                                                                       
Qwen3-8B       Ours, constrained tagging                               70.3 ± 0.4  15.7 ± 0.4  13.6 ± 0.1
               Ours, unconstrained tagging                             62.5 ± 0.1  13.8 ± 0.3  13.3 ± 0.8
               Semin et al. TAG (unconstrained)                        73.3 ± 0.3  27.2 ± 0.6  10.2 ± 0.6
               Semin et al. best method (json)                         74.1 ± 0.9         NaN         NaN
               Semin et al. best method (json_occurrence)                     NaN         NaN  11.9 ± 0.4
               Semin et al. best method (xml)                                 NaN  27.2 ± 0.6         NaN
gemma-4-31B-it Ours, constrained tagging

## 5. Error analysis

### 5.1 Outcome of every generation

In [13]:
# One outcome per generation, as in Semin et al.'s error analysis. Batch size 1 only, so a generation is one example.
# Context-based uses their taxonomy unchanged; the tagging methods use the matching labels from `outcome()` above.
OUTCOMES = ["success", "empty_prediction", "empty_response", "format_error", "format_max_tokens_exceeded",
            "span_not_found", "partial_span_not_found", "invalid_label", "partial_invalid_label", "empty_label",
            "partial_empty_label", "wrong_text", "max_tokens_exceeded", "reasoning_unterminated", "reasoning_skipped"]
assert set(preds.outcome.unique()) <= set(OUTCOMES), set(preds.outcome.unique()) - set(OUTCOMES)

# The arms that exist for all three methods -- the fair set for comparing the methods with each other.
shared = preds.query("approach == 'CTX' and bs == 1")[["dataset", "model", "arm"]].drop_duplicates()
single = preds.query("bs == 1").merge(shared, on=["dataset", "model", "arm"])


def outcome_table(df, by):
    """Share of generations (%) that ended in each outcome, per group."""
    share = df.groupby(by + ["outcome"], observed=True).size().unstack(fill_value=0)
    share = share.div(share.sum(axis=1), axis=0) * 100
    return share.reindex(columns=[o for o in OUTCOMES if o in share.columns])


print("Outcome of a generation, % of generations, shared arms, per dataset and method:")
print(order(outcome_table(single, ["dataset", "mode"]).reset_index()).set_index(["dataset", "mode"]).round(2).to_string())

# Every dataset weighs the same in the summary row, otherwise UNER (3,702 examples) would dominate it.
overall = outcome_table(single, ["dataset", "mode"]).groupby("mode").mean().reindex(list(METHODS))
print("\nAveraged over the six datasets:")
print(overall.round(2).to_string())

Outcome of a generation, % of generations, shared arms, per dataset and method:
outcome                    success  empty_prediction  empty_response  format_error  span_not_found  partial_span_not_found  invalid_label  partial_invalid_label  wrong_text  reasoning_skipped
dataset     mode                                                                                                                                                                               
CoNLL       constrained      88.45             11.55            0.00          0.00            0.00                    0.00           0.00                   0.00        0.00               0.00
            unconstrained    48.89             10.09            0.01          0.00            0.00                    0.00           0.00                   0.00       41.00               0.01
            context          82.11             14.03            0.00          0.23            1.11                    2.52           0.00               

### 5.2 The three error rates of Semin et al., per method

In [14]:
# The three error rates Semin et al. report next to F1, and what they are for each method:
#   span matching  -- a generated span is not found in the input. Context-based: proposed items whose context/entity
#                     could not be located. Tagging reproduces the text, so the same failure shows up as wrong text.
#   label matching -- a generated label is not in the label set.
#   parsing        -- the output cannot be used at all (format error, empty, or cut off at the token cap).
UNUSABLE = ["format_error", "format_max_tokens_exceeded", "empty_response",                          # context-based
            "wrong_text", "max_tokens_exceeded", "reasoning_unterminated", "reasoning_skipped"]        # tagging


def failure_modes(df, mode):
    ctx = mode == "context"
    items = df.ctx_items.sum()
    return {
        "span_not_in_input": 100 * df.ctx_not_found.sum() / items if ctx else np.nan,
        "label_not_in_schema": 100 * df.ctx_invalid_label.sum() / items if ctx else np.nan,
        "unusable_output": 100 * df.outcome.isin(UNUSABLE).mean(),
        "wrong_text": np.nan if ctx else 100 * df.wrong_text.mean(),
        "hit_token_cap": 100 * df.hit_cap.mean(),
        "reasoning_unterminated": 100 * df.unterminated.mean(),
    }


modes = pd.DataFrame({key: failure_modes(df, key[1]) for key, df in single.groupby(["dataset", "mode"], observed=True)}).T
modes.index.names = ["dataset", "mode"]
print("Failure modes in %, shared arms, batch size 1. span/label rows: % of proposed items; the rest: % of generations")
print(order(modes.reset_index()).set_index(["dataset", "mode"]).round(3).to_string())

error_modes = modes.groupby("mode").mean().reindex(list(METHODS))   # unweighted mean over the six datasets
print("\nAveraged over the six datasets:")
print(error_modes.round(3).to_string())

Failure modes in %, shared arms, batch size 1. span/label rows: % of proposed items; the rest: % of generations
                           span_not_in_input  label_not_in_schema  unusable_output  wrong_text  hit_token_cap  reasoning_unterminated
dataset     mode                                                                                                                     
CoNLL       constrained                  NaN                  NaN            0.000       0.000            0.0                   0.000
            unconstrained                NaN                  NaN           41.023      41.023            0.0                   0.000
            context                    2.313                0.000            0.231         NaN            0.0                   0.000
ToxicSpans  constrained                  NaN                  NaN            0.000       0.000            0.0                   0.000
            unconstrained                NaN                  NaN           30.833  

### 5.3 Wrong text under the constraint

In [15]:
# Wrong text under the constraint. The FSM only ever allows the next input byte or a tag, so inside the answer a
# constrained output can differ from the input in one way only: it STOPS early. This cell lists every constrained
# generation with wrong text and assigns it a cause:
#   budget in reasoning -- 18,000 tokens spent before the reasoning ended, no answer at all
#   stopped in reasoning -- the model ended the generation inside its reasoning block, no answer at all
#   budget in answer    -- reasoning terminated, the copy was cut off by the token cap
#   reasoning skipped   -- the model answered without opening a reasoning block
#   unexplained         -- none of the above: would be a real violation, look at it by hand
cons = preds.query("mode == 'constrained'")
wt = cons[cons.wrong_text == 1].copy()
wt["cause"] = np.select(
    [(wt.unterminated == 1) & (wt.hit_cap == 1), wt.unterminated == 1, wt.hit_cap == 1, wt.skipped == 1],
    ["budget in reasoning", "stopped in reasoning", "budget in answer", "reasoning skipped"], "unexplained")
print(f"constrained generations: {len(cons):,}   with wrong text: {len(wt)} ({100 * len(wt) / len(cons):.3f}%)")
print(wt.cause.value_counts().to_string())

by_arm = (wt.groupby(["dataset", "model", "arm", "bs", "cause"], observed=True)
          .agg(wrong_text=("wrong_text", "size"), mean_reasoning_tokens=("reasoning_tokens", "mean"),
               mean_answer_tokens=("answer_tokens", "mean")).reset_index())
by_arm = by_arm.merge(cons.groupby(["dataset", "model", "arm", "bs"], observed=True).size().rename("generations").reset_index())
by_arm["wrong_text_%"] = 100 * by_arm.wrong_text / by_arm.generations
print()
print(order(by_arm).round(2).to_string(index=False))

# Reasoning arms where the model skipped the reasoning block: how often, and how often the copy is then wrong.
skipped = cons[cons.skipped == 1].groupby(["dataset", "model", "arm", "bs"], observed=True).agg(
    skipped=("skipped", "size"), wrong_text=("wrong_text", "sum")).reset_index()
print("\ngenerations with skipped reasoning, and how many of them have wrong text:")
print(order(skipped).to_string(index=False))

wt[wt.cause == "unexplained"][["dataset", "subset", "model", "arm", "bs", "seed", "key", "out_tokens",
                               "reasoning_tokens", "answer_tokens"]]

constrained generations: 255,312   with wrong text: 73 (0.029%)
cause
budget in answer        28
budget in reasoning     21
reasoning skipped       17
stopped in reasoning     7

    dataset          model    arm  bs                cause  wrong_text  mean_reasoning_tokens  mean_answer_tokens  generations  wrong_text_%
 ToxicSpans    gpt-oss-20b medium   1 stopped in reasoning           2                 199.00                0.00         3000          0.07
LegalQAEval gemma-4-E2B-it     ON   1  budget in reasoning           2               18000.00                0.00         3618          0.06
LegalQAEval    gpt-oss-20b medium   1 stopped in reasoning           1                 284.00                0.00         3618          0.03
       UNER gemma-4-E2B-it     ON   1 stopped in reasoning           1                 320.00                0.00        11106          0.01
       UNER    Qwen3.8-27B    low  64     budget in answer          15               15438.47             2561.53   

,dataset,subset,model,arm,bs,seed,key,out_tokens,reasoning_tokens,answer_tokens


### 5.4 Token budget and reasoning termination

In [16]:
# Token budget and termination per reasoning arm, constrained tagging. Non-termination and answer truncation are the
# only two ways the constrained output fails, and both are a function of how much the model reasons.
budget = (cons.groupby(["dataset", "model", "arm", "bs"], observed=True)
          .agg(generations=("seed", "size"), unterminated=("unterminated", "mean"), skipped=("skipped", "mean"),
               hit_cap=("hit_cap", "mean"), wrong_text=("wrong_text", "mean"), reasoning_tokens=("reasoning_tokens", "mean"),
               answer_tokens=("answer_tokens", "mean"))
          .reset_index())
budget[["unterminated", "skipped", "hit_cap", "wrong_text"]] *= 100
print("Constrained tagging, reasoning arms only: % of generations and mean tokens per generation")
print(order(budget.query("arm != 'OFF'")).round(2).to_string(index=False))

# The same, per model and arm over all datasets (each dataset weighs the same) -- the termination-rate table.
print("\nPer model and arm, averaged over datasets (batch size 1):")
print(order(budget.query("bs == 1").groupby(["model", "arm"], observed=True)
            [["unterminated", "skipped", "hit_cap", "wrong_text", "reasoning_tokens", "answer_tokens"]].mean().reset_index())
      .round(2).to_string(index=False))

Constrained tagging, reasoning arms only: % of generations and mean tokens per generation
    dataset          model    arm  bs  generations  unterminated  skipped  hit_cap  wrong_text  reasoning_tokens  answer_tokens
      CoNLL gemma-4-E2B-it     ON   1         3456          0.00     0.00     0.00        0.00            391.48          44.22
      CoNLL gemma-4-E2B-it     ON  64           54          0.00     0.00     0.00        0.00           2501.78        1516.83
      CoNLL    Qwen3.8-27B    low   1         3456          0.00     0.00     0.00        0.00            163.58          43.09
      CoNLL    Qwen3.8-27B    low  64           60          0.00     0.00     0.00        0.00           3370.83        2564.85
      CoNLL    gpt-oss-20b    low   1         3456          0.00     0.00     0.00        0.00             93.39          43.10
      CoNLL    gpt-oss-20b    low  64           54          0.00     0.00     0.00        0.00             32.93        1519.54
      CoNLL   

### 5.5 Span-level error decomposition

In [17]:
# What is wrong with the spans that were predicted. Every predicted span is one of: exact / right place wrong label /
# right label wrong boundaries / wrong both / spurious (touches no gold span). Gold spans nothing touches are missed.
E_PRED = ["e_exact", "e_label", "e_boundary", "e_bound_label", "e_spurious"]
decomp = single.groupby(["dataset", "mode"], observed=True)[E_PRED + ["e_missed", "n_pred", "n_gold"]].sum()
for col in E_PRED:
    decomp[col.replace("e_", "") + " %pred"] = 100 * decomp[col] / decomp.n_pred
decomp["missed %gold"] = 100 * decomp.e_missed / decomp.n_gold
decomp["pred/gold spans"] = decomp.n_pred / decomp.n_gold
print("Span-level error decomposition, shared arms, batch size 1:")
print(order(decomp.drop(columns=E_PRED + ["e_missed"]).reset_index()).set_index(["dataset", "mode"]).round(2).to_string())

Span-level error decomposition, shared arms, batch size 1:
                           n_pred  n_gold  exact %pred  label %pred  boundary %pred  bound_label %pred  spurious %pred  missed %gold  pred/gold spans
dataset     mode                                                                                                                                     
CoNLL       constrained     26007   22728        55.70        10.98           13.01               3.88           16.43          3.01             1.14
            unconstrained   14044   22728        63.45        11.99            4.75               2.61           17.20         48.66             0.62
            context         24535   22728        68.71        10.57            3.62               2.95           14.16          7.68             1.08
ToxicSpans  constrained     14034    6924        16.13         0.00           15.75               0.00           68.13         33.02             2.03
            unconstrained    8141    6924

### 5.6 Label confusion

In [18]:
# Label confusion on spans whose boundaries are exactly right -- only meaningful on the multi-label datasets. Rows are
# gold labels, columns predicted labels, each row in % of that gold label. This is what separates hard from soft F1.
for dataset in ["CoNLL", "UNER", "ESA-MT", "GEC"]:
    c = confusion.query("dataset == @dataset and mode == 'constrained' and bs == 1")
    table = c.pivot_table(index="gold_label", columns="pred_label", values="n", aggfunc="sum", fill_value=0)
    print(f"\n{dataset}: constrained tagging, all arms, {int(table.values.sum()):,} boundary-exact spans")
    print((table.div(table.sum(axis=1), axis=0) * 100).round(1).to_string())


CoNLL: constrained tagging, all arms, 45,682 boundary-exact spans
pred_label   LOC  MISC   ORG   PER
gold_label                        
LOC         96.3   1.3   2.2   0.2
MISC        11.3  79.6   6.9   2.2
ORG         31.7   2.9  64.6   0.9
PER          0.7   1.3   0.6  97.3

UNER: constrained tagging, all arms, 151,806 boundary-exact spans
pred_label   LOC   ORG   PER
gold_label                  
LOC         97.2   2.2   0.6
ORG         10.6  87.5   1.9
PER          1.2   1.2  97.6

ESA-MT: constrained tagging, all arms, 3,932 boundary-exact spans
pred_label  MAJOR  MINOR
gold_label              
MAJOR        56.7   43.3
MINOR        71.8   28.2

GEC: constrained tagging, all arms, 46,888 boundary-exact spans
pred_label      M     R     U
gold_label                   
M           100.0   0.0   0.0
R             0.0  93.7   6.3
U             0.0  20.6  79.4


### 5.7 Span length and count against gold

In [19]:
# Span length and count against gold. Semin et al. report the mean span length too (GEC: reference 59.2 characters,
# their TAG 42.4, their MATCH 198.0); it shows whether a method localises the error or over-marks.
lengths = single.groupby(["dataset", "mode"], observed=True)[["gold_chars", "pred_chars", "n_gold", "n_pred",
                                                             "gold_empty", "pred_empty", "n_examples"]].sum()
lengths["gold_span_len"] = lengths.gold_chars / lengths.n_gold
lengths["pred_span_len"] = lengths.pred_chars / lengths.n_pred
lengths["gold_spans/ex"] = lengths.n_gold / lengths.n_examples
lengths["pred_spans/ex"] = lengths.n_pred / lengths.n_examples
# Zero-length spans only exist on MultiGEC (label M, a missing word): how much of that convention does a method use?
lengths["gold_empty_%"] = 100 * lengths.gold_empty / lengths.n_gold
lengths["pred_empty_%"] = 100 * lengths.pred_empty / lengths.n_pred
cols = ["gold_span_len", "pred_span_len", "gold_spans/ex", "pred_spans/ex", "gold_empty_%", "pred_empty_%"]
print("Mean span length in characters and spans per example, shared arms, batch size 1:")
print(order(lengths[cols].reset_index()).set_index(["dataset", "mode"]).round(2).to_string())

Mean span length in characters and spans per example, shared arms, batch size 1:
                           gold_span_len  pred_span_len  gold_spans/ex  pred_spans/ex  gold_empty_%  pred_empty_%
dataset     mode                                                                                                 
CoNLL       constrained             9.47          11.41           1.64           1.88          0.00          0.00
            unconstrained           9.47           9.72           1.64           1.02          0.00          0.00
            context                 9.47           9.82           1.64           1.77          0.00          0.00
ToxicSpans  constrained             6.93          23.76           0.58           1.17          0.00          0.00
            unconstrained           6.93          15.48           0.58           0.68          0.00          0.01
            context                 6.93          12.04           0.58           0.93          0.00          0.00
LegalQA

## 6. Reasoning and decode-time cost

In [20]:
# What reasoning buys under the constraint: the same model with reasoning on vs off (gemma-4-E2B: ON vs OFF,
# Qwen3.8-27B: low vs OFF) and, for gpt-oss which cannot switch reasoning off, medium vs low effort. Hard F1, bs 1.
PAIRS = [("gemma-4-E2B-it", "ON", "OFF"), ("Qwen3.8-27B", "low", "OFF"),
         ("gpt-oss-20b", "medium", "low"), ("gpt-oss-120b", "medium", "low")]
f1 = final.query("mode == 'constrained' and bs == 1").set_index(["model", "arm", "dataset"])["hard_f1_mean"] * 100
rows = []
for model, more, less in PAIRS:
    for dataset in DATASETS:
        rows.append({"model": model, "comparison": f"{more} - {less}", "dataset": dataset,
                     "more": f1[model, more, dataset], "less": f1[model, less, dataset]})
reasoning = pd.DataFrame(rows)
reasoning["delta"] = reasoning["more"] - reasoning["less"]

wide = reasoning.pivot(index=["model", "comparison"], columns="dataset", values="delta")[DATASETS]
wide["mean"] = wide.mean(axis=1)
print("Hard F1 gain (percentage points) of the arm with MORE reasoning, constrained tagging:")
print(wide.reindex([p[0] for p in PAIRS], level=0).round(2).to_string())

Hard F1 gain (percentage points) of the arm with MORE reasoning, constrained tagging:
dataset                      CoNLL  ToxicSpans  LegalQAEval   UNER  ESA-MT    GEC  mean
model          comparison                                                              
gemma-4-E2B-it ON - OFF      11.57        0.15        -0.34   5.71    3.20   0.72  3.50
Qwen3.8-27B    low - OFF      7.64        2.61         9.94  -5.42    0.26  25.38  6.73
gpt-oss-20b    medium - low   4.83       -3.49         4.95  10.48    0.58   7.63  4.16
gpt-oss-120b   medium - low   1.14       -1.66         4.89   9.75    0.49   1.18  2.63


In [21]:
# Decode-time cost of the constraint. Both decoding modes of an arm ran in the same job on the same GPU(s), one after
# the other, so their ratio is a fair comparison even though absolute tokens/s depends on how many processes shared
# the GPU. tokens/s = all generated tokens / all generation seconds of the arm.
cg = preds.query("approach == 'CG' and bs == 1")
speed = cg.groupby(["dataset", "model", "arm", "mode"], observed=True).agg(
    tokens=("out_tokens", "sum"), seconds=("gen_seconds", "sum"),
    answer_tokens=("answer_tokens", "mean"), out_tokens=("out_tokens", "mean"))
speed["tok_per_s"] = speed.tokens / speed.seconds
speed = speed[["tok_per_s", "answer_tokens", "out_tokens"]].unstack("mode")
speed["overhead_%"] = 100 * (1 - speed["tok_per_s", "constrained"] / speed["tok_per_s", "unconstrained"])

efficiency = speed.groupby("model").mean().reindex(MODELS)     # every dataset and arm weighs the same
efficiency.loc["ALL MODELS"] = efficiency.mean()
print("tokens/s, mean answer tokens and mean output tokens per generation; overhead = throughput lost to the constraint")
print(efficiency.round(1).to_string())

tokens/s, mean answer tokens and mean output tokens per generation; overhead = throughput lost to the constraint
                 tok_per_s               answer_tokens                out_tokens               overhead_%
mode           constrained unconstrained   constrained unconstrained constrained unconstrained           
model                                                                                                    
gemma-4-E2B-it        12.5          12.3         109.1          85.1       436.4         412.2       -2.2
gemma-4-31B-it        25.7          25.8         126.9         126.1       126.9         126.1        0.3
Qwen3-8B              16.7          17.1         120.6         125.4       120.6         125.4        2.4
Qwen3.8-27B           18.2          18.0         123.3         120.3       483.2         479.8       -1.1
gpt-oss-20b           21.4          21.0         123.7         122.0       872.3         882.8       -2.2
gpt-oss-120b          25.2          25.

## 7. LaTeX rows for `paper/tables/`

Each cell prints the body rows of one table, ready to paste over the `\stub` rows.

In [22]:
# ---- paper/tables/tab_main_results.tex -------------------------------------------------------------------------
# Cell = hard F1 (soft F1), mean over the 3 seeds, batch size 1. One arm per model for the "ours" block: reasoning off
# where the model has an off arm, effort=low for gpt-oss (it has none). Change MAIN_ARM to report another arm.
MAIN_ARM = {"gemma-4-E2B-it": "OFF", "gemma-4-31B-it": "OFF", "Qwen3-8B": "OFF", "Qwen3.8-27B": "OFF",
            "gpt-oss-20b": "low", "gpt-oss-120b": "low"}


def tex_row(name, mode, model, arm, bs=1):
    """One LaTeX row over the six datasets, in the column order of the table."""
    cells = []
    for dataset in DATASETS:
        r = final.query("dataset == @dataset and mode == @mode and model == @model and arm == @arm and bs == @bs")
        cells.append("---" if r.empty else f"{100 * r.hard_f1_mean.item():.1f} ({100 * r.soft_f1_mean.item():.1f})")
    return f"{name:<34} & " + " & ".join(cells) + r" \\"


print(r"\multicolumn{7}{l}{\emph{Constrained tagging (ours, no training)}} \\")
for model, arm in MAIN_ARM.items():
    print(tex_row(model, "constrained", model, arm))

print(r"\midrule")
print(r"\multicolumn{7}{l}{\emph{Prompting (context-based)}} \\")
for model, arm in order(shared[["model", "arm"]].drop_duplicates()).itertuples(index=False):
    print(tex_row(f"{model} ({arm})", "context", model, arm))

print(r"\midrule")
print(r"\multicolumn{7}{l}{\emph{Unconstrained decoding (same prompt, constraint off)}} \\")
for model, arm in MAIN_ARM.items():
    print(tex_row(model, "unconstrained", model, arm))

# The remaining "ours" arms (reasoning on / higher effort), in case one of them should replace a row above.
print("\n% other constrained arms:")
for model in MODELS:
    for arm in ARMS:
        if arm != MAIN_ARM[model] and not final.query("model == @model and arm == @arm and mode == 'constrained'").empty:
            print("% " + tex_row(f"{model} ({arm})", "constrained", model, arm))

\multicolumn{7}{l}{\emph{Constrained tagging (ours, no training)}} \\


gemma-4-E2B-it                     & 56.4 (76.5) & 13.2 (13.2) & 11.2 (11.2) & 58.6 (66.3) & 4.6 (9.8) & 8.2 (12.8) \\
gemma-4-31B-it                     & 84.7 (94.6) & 25.3 (25.3) & 55.5 (55.5) & 88.7 (92.2) & 14.0 (34.0) & 50.7 (54.0) \\
Qwen3-8B                           & 63.3 (78.3) & 28.7 (28.7) & 19.6 (19.6) & 70.3 (76.3) & 13.6 (20.5) & 15.7 (20.2) \\
Qwen3.8-27B                        & 72.8 (81.5) & 17.4 (17.4) & 41.8 (41.8) & 59.8 (63.2) & 9.2 (22.8) & 24.5 (31.1) \\
gpt-oss-20b                        & 69.9 (81.9) & 20.5 (20.5) & 42.9 (42.9) & 75.7 (80.4) & 12.9 (25.7) & 24.5 (28.1) \\
gpt-oss-120b                       & 78.6 (89.6) & 20.6 (20.6) & 47.9 (47.9) & 79.5 (83.1) & 11.8 (28.2) & 31.8 (36.4) \\
\midrule
\multicolumn{7}{l}{\emph{Prompting (context-based)}} \\
gemma-4-E2B-it (OFF)               & 71.3 (86.0) & 32.5 (32.5) & 38.8 (38.8) & 78.1 (82.9) & 6.3 (16.6) & 26.9 (28.8) \\


gemma-4-E2B-it (ON)                & 72.5 (85.8) & 29.4 (29.4) & --- & 84.1 (88.0) & 7.7 (18.8) & --- \\


Qwen3.8-27B (OFF)                  & 84.0 (92.7) & 21.7 (21.7) & 16.8 (16.8) & 86.9 (90.3) & 8.2 (20.0) & 40.5 (43.4) \\


gpt-oss-20b (low)                  & 73.3 (84.4) & 30.1 (30.1) & 32.6 (32.6) & 80.8 (84.8) & 11.8 (24.6) & 32.5 (35.1) \\
\midrule
\multicolumn{7}{l}{\emph{Unconstrained decoding (same prompt, constraint off)}} \\
gemma-4-E2B-it                     & 8.7 (10.8) & 28.7 (28.7) & 2.1 (2.1) & 22.2 (24.9) & 2.0 (5.0) & 0.7 (0.8) \\
gemma-4-31B-it                     & 72.4 (81.7) & 24.9 (24.9) & 56.0 (56.0) & 84.5 (87.9) & 13.9 (34.1) & 36.3 (38.5) \\
Qwen3-8B                           & 58.7 (72.6) & 32.6 (32.6) & 17.9 (17.9) & 62.5 (67.4) & 13.3 (20.0) & 13.8 (16.3) \\
Qwen3.8-27B                        & 69.9 (78.6) & 23.3 (23.3) & 46.4 (46.4) & 60.7 (64.2) & 9.1 (22.3) & 31.1 (32.7) \\
gpt-oss-20b                        & 59.8 (72.0) & 23.7 (23.7) & 40.0 (40.0) & 70.0 (73.4) & 10.2 (20.7) & 14.2 (15.3) \\
gpt-oss-120b                       & 72.9 (83.9) & 21.9 (21.9) & 46.8 (46.8) & 77.9 (81.4) & 10.4 (24.3) & 17.6 (18.5) \\

% other constrained arms:


% gemma-4-E2B-it (ON)                & 68.0 (81.9) & 13.4 (13.4) & 10.9 (10.9) & 64.3 (69.1) & 7.8 (17.3) & 9.0 (16.1) \\


% Qwen3.8-27B (low)                  & 80.4 (88.5) & 20.0 (20.0) & 51.8 (51.8) & 54.4 (56.7) & 9.5 (25.9) & 49.9 (52.7) \\
% gpt-oss-20b (medium)               & 74.7 (86.4) & 17.0 (17.0) & 47.9 (47.9) & 86.2 (90.0) & 13.5 (27.2) & 32.2 (35.8) \\
% gpt-oss-120b (medium)              & 79.7 (91.5) & 18.9 (18.9) & 52.8 (52.8) & 89.3 (92.7) & 12.3 (29.6) & 32.9 (37.1) \\


In [23]:
# ---- paper/tables/tab_published_comparison.tex -----------------------------------------------------------------
# Hard F1, reasoning off. Published = Semin et al. TAG, recomputed from their results.csv. The UNER cell of "Ours" is
# the capped run until the full-dataset run has finished -- re-run this cell after the last section then.
def published_row(name, table, model, **query):
    cells = []
    for dataset in ["UNER", "GEC", "ESA-MT"]:
        r = table[(table.model == model) & (table.dataset == dataset)]
        for col, value in query.items():
            r = r[r[col] == value]
        cells.append(f"{100 * r.hard_f1_mean.item():.1f}" if len(r) == 1 else r"\stub")
    return f"{name:<38} & " + " & ".join(cells) + r" \\"


ours_full = globals().get("final_full")   # set by the last section once the full UNER run exists
for model in ["gemma-4-31B-it", "Qwen3-8B"]:
    print(rf"\multicolumn{{4}}{{l}}{{\emph{{{model}}}}} \\")
    print(published_row("Published (TAG)", semin_f1, model, method_name="xml"))
    for mode, name in (("unconstrained", "Ours, unconstrained"), ("constrained", "Ours, constrained")):
        row = published_row(name, final.query("bs == 1 and arm == 'OFF'"), model, mode=mode)
        if ours_full is not None:   # swap in the full-dataset UNER number
            uner = ours_full.query("model == @model and mode == @mode")
            if len(uner) == 1:
                cells = row.split(" & ")
                cells[1] = f"{100 * uner.hard_f1_mean.item():.1f}"
                row = " & ".join(cells)
        print(row)
print("% UNER for ours:", "full dataset" if ours_full is not None else "CAPPED at 250 examples per treebank")

\multicolumn{4}{l}{\emph{gemma-4-31B-it}} \\
Published (TAG)                        & 88.9 & 52.9 & 14.2 \\
Ours, unconstrained                    & 84.5 & 36.3 & 13.9 \\
Ours, constrained                      & 88.7 & 50.7 & 14.0 \\
\multicolumn{4}{l}{\emph{Qwen3-8B}} \\
Published (TAG)                        & 73.3 & 27.2 & 10.2 \\
Ours, unconstrained                    & 62.5 & 13.8 & 13.3 \\
Ours, constrained                      & 70.3 & 15.7 & 13.6 \\
% UNER for ours: CAPPED at 250 examples per treebank


In [24]:
# ---- paper/tables/tab_error_modes.tex --------------------------------------------------------------------------
# % averaged over the six datasets, shared arms, batch size 1 (the `error_modes` table above). The xgrammar column
# stays a stub until that baseline is run. "Ours" is zero by construction in the first two rows, not by measurement.
em = error_modes


def cell(mode, col):
    return f"{em.loc[mode, col]:.2f}"


print(rf"Span not in input   & {cell('context', 'span_not_in_input')} & \stub & $0$ (struct.) \\")
print(rf"Label not in schema & {cell('context', 'label_not_in_schema')} & \stub & $0$ (struct.) \\")
print(rf"Output unparseable / \wtr{{}} & {cell('context', 'unusable_output')} & \stub & {cell('constrained', 'wrong_text')} \\")
print(r"\midrule")
print(rf"Reasoning non-termination & {cell('context', 'reasoning_unterminated')} & --- & {cell('constrained', 'reasoning_unterminated')} \\")
print(f"% unconstrained tagging for reference: wrong text {cell('unconstrained', 'wrong_text')}, "
      f"unusable output {cell('unconstrained', 'unusable_output')}")

# Over ALL constrained arms (not only the shared ones, batch size 64 included) -- the numbers for the text:
print(f"% all {len(cons):,} constrained generations: wrong text {100 * cons.wrong_text.mean():.3f}%, "
      f"reasoning non-termination {100 * cons.unterminated.mean():.3f}%, token cap {100 * cons.hit_cap.mean():.3f}%")

Span not in input   & 9.39 & \stub & $0$ (struct.) \\
Label not in schema & 0.01 & \stub & $0$ (struct.) \\
Output unparseable / \wtr{} & 0.63 & \stub & 0.00 \\
\midrule
Reasoning non-termination & 0.02 & --- & 0.00 \\
% unconstrained tagging for reference: wrong text 39.90, unusable output 39.90
% all 255,312 constrained generations: wrong text 0.029%, reasoning non-termination 0.011%, token cap 0.019%


In [25]:
# ---- paper/tables/tab_reasoning.tex ----------------------------------------------------------------------------
# Hard F1 averaged over the six datasets, constrained tagging, batch size 1. ON is the reasoning arm that was run for
# the model (gemma-4-E2B: default, Qwen3.8-27B and gpt-oss: effort=low). gemma-4-31B and Qwen3-8B were only run with
# reasoning off, so their ON cell is n/a rather than a stub.
ON_ARM = {"gemma-4-E2B-it": "ON", "Qwen3.8-27B": "low", "gpt-oss-20b": "low", "gpt-oss-120b": "low"}
avg = (final.query("mode == 'constrained' and bs == 1").groupby(["model", "arm"], observed=True)["hard_f1_mean"].mean() * 100)
for model in MODELS:
    on = avg.get((model, ON_ARM.get(model)), np.nan)
    off = avg.get((model, "OFF"), np.nan)
    fmt = lambda v: "n/a" if np.isnan(v) else f"{v:.1f}"
    delta = "---" if np.isnan(on) or np.isnan(off) else f"{on - off:+.1f}"
    print(f"{model:<14} & {fmt(on)} & {fmt(off)} & {delta} \\\\")

# gpt-oss has no off arm; its reasoning comparison is low vs medium effort.
for model in ["gpt-oss-20b", "gpt-oss-120b"]:
    print(f"% {model}: low {avg[model, 'low']:.1f}, medium {avg[model, 'medium']:.1f}, "
          f"medium - low {avg[model, 'medium'] - avg[model, 'low']:+.1f}")

gemma-4-E2B-it & 28.9 & 25.4 & +3.5 \\
gemma-4-31B-it & n/a & 53.1 & --- \\
Qwen3-8B       & n/a & 35.2 & --- \\
Qwen3.8-27B    & 44.3 & 37.6 & +6.7 \\
gpt-oss-20b    & 41.1 & n/a & --- \\
gpt-oss-120b   & 45.0 & n/a & --- \\
% gpt-oss-20b: low 41.1, medium 45.2, medium - low +4.2
% gpt-oss-120b: low 45.0, medium 47.7, medium - low +2.6


In [26]:
# ---- paper/tables/tab_efficiency.tex ---------------------------------------------------------------------------
# From the `efficiency` table above (all models, datasets and arms weighed equally). xgrammar and the whole-sequence
# processor were not part of these runs, their rows stay stubs.
e = efficiency.loc["ALL MODELS"]
print(rf"Unconstrained        & {e['tok_per_s', 'unconstrained']:.1f} & --- & {e['answer_tokens', 'unconstrained']:.0f} \\")
print(r"xgrammar             & \stub & \stub & \stub \\")
print(r"Ours (\ws{})    & \stub & \stub & \stub \\")
print(rf"Ours (\ta{{}})    & {e['tok_per_s', 'constrained']:.1f} & {e['overhead_%'].item():.1f}\% & {e['answer_tokens', 'constrained']:.0f} \\")

Unconstrained        & 19.9 & --- & 119 \\
xgrammar             & \stub & \stub & \stub \\
Ours (\ws{})    & \stub & \stub & \stub \\
Ours (\ta{})    & 20.0 & -0.6\% & 124 \\


## 8. Full-dataset UNER runs for the Semin et al. comparison

In [27]:
# Full UNER test sets (7,523 examples, no cap), gemma-4-31B and Qwen3-8B, reasoning off, 3 seeds -- the protocol of
# Semin et al. apart from the seed count (they use 5). Written to Experiment_results_publication_full/ so the capped
# runs are untouched. Nothing happens here until both jobs have finished.
UNER_FULL_N = 7523
if not (RES_FULL / "UNER" / "Constrained-Gen" / "Predictions").is_dir():
    print("full-dataset runs not there yet")
else:
    preds_full, _ = load_predictions(RES_FULL, "full", datasets=["UNER"], refresh=True)
    n = preds_full.groupby(["model", "mode", "seed"]).n_examples.sum()
    print("examples per model / mode / seed (each must be 7,523):")
    print(n.unstack("seed").to_string())

    if (n == UNER_FULL_N).all() and len(n) == 2 * 2 * 3:
        _, _, final_full = aggregate(preds_full)
        theirs = semin_f1.query("dataset == 'UNER' and method_name == 'xml'").set_index("model")
        for r in order(final_full).itertuples():
            print(f"{r.model:<16} {r.mode:<14} hard F1 {pm(r.hard_f1_mean, r.hard_f1_std)}   soft F1 {pm(r.soft_f1_mean, r.soft_f1_std)}"
                  f"   entity F1 {pm(r.ent_f1_mean, r.ent_f1_std)}   | Semin et al. TAG hard F1 {100 * theirs.hard_f1_mean[r.model]:.2f}")
        print("\nnow re-run the tab_published_comparison cell")
    else:
        print("\nruns still incomplete -- no scores computed")

full-dataset runs not there yet
